# Unfold studied MC with a controlled-sample response

Build a flattened $p_T^{ave}$–$\eta_{CM}$ response from the 60% controlled MC subsample, unfold the independent 40% studied reconstructed distribution, and compare the result with studied generator truth. The controlled response, misses, fakes, and training marginals use the same default-JER unfolding definition.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'CMakeLists.txt').is_file()
                     and (p / 'hist_analysis').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Start Jupyter from the jetAnalysis repository root')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root, load_roounfold
ROOT = load_root(batch=True)
from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import DIJET_DELTA_PHI_SELECTION_LABEL
from hist_analysis.python.histogram_io import resolve_combined_file, resolve_direction_file
from hist_analysis.python.root_style import (
    DEFAULT_PLOT_STYLE, save_canvas, set_2d_style, set_legend_style,
    set_pad_style, set_unfolding_1d_style,
)
ROOUNFOLD_ROOT, ROOUNFOLD_LIBRARY = load_roounfold(ROOT, project_root=PROJECT_ROOT)
import RooUnfold
ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

from hist_analysis.config.histograms import DIJET_PTAVE_BINS, TEST_DIJET_PTAVE_BINS
from hist_analysis.python.unfolding import (
    UnfoldingInputKeys, as_pt_intervals, build_roounfold_response,
    calculate_response_diagnostics, flatten_pt_eta_projections,
    flatten_sparse_response, load_unfolding_inputs, project_eta_by_pt,
    project_response_eta_blocks, unfold_bayes, write_unfolding_output,
)
from hist_analysis.python.unfolding_plots import (
    draw_flattened_response, draw_projection_response,
    draw_unfolding_classification, draw_unfolding_closure,
    draw_unfolding_closure_by_pt,
)


## Configuration

The response-training objects all come from the controlled sample. The measured spectrum and closure truth both come from the studied sample. Intervals are half-open. `RESPONSE_SCALE` protects small weighted response entries from RooUnfold's absolute sanitization threshold.


In [ ]:
GENERATOR = 'embedding'       # embedding or pythia
DIRECTION = 'Pbgoing'        # pgoing, Pbgoing, or combined
FILE_STEM = 'jetId'
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.3, 2.4, 3.0)
ETA_CUT_INDEX = 5
PTAVE_BIN_SET = 'test'  # test or standard
PTAVE_BIN_SETS = {'test': TEST_DIJET_PTAVE_BINS, 'standard': DIJET_PTAVE_BINS}
PT_AVE_BINS = tuple(PTAVE_BIN_SETS[PTAVE_BIN_SET])
N_ITERATIONS = 4
RESPONSE_SCALE = 1.0e12
PLOT_MISS_AND_FAKES = False
SAVE_PNG = False

CONTROLLED_TRUTH_TEMPLATE = 'hGenDijetPtEtaCMControlledSample_{eta_cut_index}'
CONTROLLED_MEASURED_TEMPLATE = 'hRecoDijetPtEtaCMJerDefUnfoldControlledSample_{eta_cut_index}'
STUDIED_TRUTH_TEMPLATE = 'hGenDijetPtEtaCMStudiedSample_{eta_cut_index}'
STUDIED_MEASURED_TEMPLATE = 'hRecoDijetPtEtaCMJerDefUnfoldStudiedSample_{eta_cut_index}'
CONTROLLED_RESPONSE_TEMPLATE = 'hGenDijetPtEtaCMVsRecoPtEtaCMControlledSample_{eta_cut_index}'
CONTROLLED_MISS_TEMPLATE = 'hGenDijetPtEtaCMMissControlledSample_{eta_cut_index}'
CONTROLLED_FAKE_TEMPLATE = 'hRecoDijetPtEtaCMFakeControlledSample_{eta_cut_index}'

if PTAVE_BIN_SET not in PTAVE_BIN_SETS:
    raise ValueError(f'Unsupported PTAVE_BIN_SET={PTAVE_BIN_SET!r}')
if GENERATOR not in ('embedding', 'pythia'):
    raise ValueError(f'Unsupported GENERATOR={GENERATOR!r}')
if DIRECTION not in ('pgoing', 'Pbgoing', 'combined'):
    raise ValueError(f'Unsupported DIRECTION={DIRECTION!r}')
if ETA_CUT_INDEX < 0 or ETA_CUT_INDEX >= len(ETA_CUTS):
    raise IndexError(f'Invalid ETA_CUT_INDEX={ETA_CUT_INDEX}')
if RESPONSE_SCALE <= 0.0:
    raise ValueError('RESPONSE_SCALE must be positive')
if not isinstance(PLOT_MISS_AND_FAKES, bool):
    raise TypeError('PLOT_MISS_AND_FAKES must be True or False')

if DIRECTION == 'combined':
    INPUT_FILE = resolve_combined_file(BASE_DIR, GENERATOR, FILE_STEM)
else:
    INPUT_FILE = resolve_direction_file(BASE_DIR, GENERATOR, DIRECTION, FILE_STEM)
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]
ETA_TAG = f'{ETA_CUT:g}'.replace('.', 'p')
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_UNFOLD2D_STUDIED_CONTROLLED_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'unfold2D_studied_to_controlled',
))
OUTPUT_TAG = (f'{GENERATOR}_{DIRECTION}_unfold2D_studiedToControlled_'
              f'eta_{ETA_TAG}_iter_{N_ITERATIONS}')
OUTPUT_ROOT_FILE = OUTPUT_DIR / f'{OUTPUT_TAG}.root'

PT_AVE_INTERVALS = as_pt_intervals(PT_AVE_BINS)


## Load controlled training objects and studied closure objects


In [ ]:
controlled_keys = UnfoldingInputKeys(
    truth=CONTROLLED_TRUTH_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX), measured=CONTROLLED_MEASURED_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX),
    response=CONTROLLED_RESPONSE_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX), miss=CONTROLLED_MISS_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX),
    fake=CONTROLLED_FAKE_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX),
)
controlled_inputs = load_unfolding_inputs(INPUT_FILE, controlled_keys)
hControlledTruth2D, hControlledMeasured2D = controlled_inputs.truth, controlled_inputs.measured
hControlledResponseSparse, hControlledMiss2D, hControlledFake2D = controlled_inputs.response, controlled_inputs.miss, controlled_inputs.fake
from hist_analysis.python.histogram_io import load_histogram
hStudiedTruth2D = load_histogram(str(INPUT_FILE), STUDIED_TRUTH_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX))
hStudiedMeasured2D = load_histogram(str(INPUT_FILE), STUDIED_MEASURED_TEMPLATE.format(eta_cut_index=ETA_CUT_INDEX))
print(INPUT_FILE); print(controlled_keys)

## Project pTave intervals and construct flattened inputs

Global bins are ordered as `[pTave block][eta bin]`. All response pT migration blocks are retained.


In [ ]:
controlledTruthByPt = project_eta_by_pt(hControlledTruth2D, PT_AVE_INTERVALS, name_prefix='hControlledTruthEtaCM')
controlledMeasuredByPt = project_eta_by_pt(hControlledMeasured2D, PT_AVE_INTERVALS, name_prefix='hControlledMeasuredEtaCM')
studiedTruthByPt = project_eta_by_pt(hStudiedTruth2D, PT_AVE_INTERVALS, name_prefix='hStudiedTruthEtaCM')
studiedMeasuredByPt = project_eta_by_pt(hStudiedMeasured2D, PT_AVE_INTERVALS, name_prefix='hStudiedMeasuredEtaCM')
controlledMissByPt = project_eta_by_pt(hControlledMiss2D, PT_AVE_INTERVALS, name_prefix='hControlledMissEtaCM')
controlledFakeByPt = project_eta_by_pt(hControlledFake2D, PT_AVE_INTERVALS, name_prefix='hControlledFakeEtaCM')
hControlledTruth, layout = flatten_pt_eta_projections(controlledTruthByPt, name='hControlledTruthEtaCM', pt_bins=PT_AVE_INTERVALS)
hControlledMeasured, _ = flatten_pt_eta_projections(controlledMeasuredByPt, name='hControlledMeasuredEtaCM', layout=layout)
hStudiedTruth, _ = flatten_pt_eta_projections(studiedTruthByPt, name='hStudiedTruthEtaCM', layout=layout)
hStudiedMeasured, _ = flatten_pt_eta_projections(studiedMeasuredByPt, name='hStudiedMeasuredEtaCM', layout=layout)
hControlledMiss, _ = flatten_pt_eta_projections(controlledMissByPt, name='hControlledMissEtaCM', layout=layout)
hControlledFake, _ = flatten_pt_eta_projections(controlledFakeByPt, name='hControlledFakeEtaCM', layout=layout)
hControlledResponse, _ = flatten_sparse_response(hControlledResponseSparse, PT_AVE_INTERVALS, name='hControlledResponseEtaCM', layout=layout)
n_pt,n_eta,n_global=layout.n_pt_bins,layout.n_eta_bins,layout.n_global_bins
print(f'nPt={n_pt}, nEta={n_eta}, nGlobal={n_global}')

## Construct the controlled response and unfold studied reco


In [ ]:
diagnostics = calculate_response_diagnostics(hControlledResponse,hControlledTruth,hControlledMeasured,explicit_miss=hControlledMiss,explicit_fake=hControlledFake)
hMatchedControlledTruth,hMatchedControlledReco=diagnostics.matched_truth,diagnostics.matched_measured
hEffectiveControlledMiss,hEffectiveControlledFake=diagnostics.effective_miss,diagnostics.effective_fake
hBoundaryControlledMiss,hBoundaryControlledFake=diagnostics.boundary_miss,diagnostics.boundary_fake
response_bundle=build_roounfold_response(RooUnfold,hControlledTruth,hControlledMeasured,hControlledResponse,diagnostics=diagnostics,scale=RESPONSE_SCALE,require_fakes=True,name='controlledResponseEtaCM',title='Controlled-sample flattened dijet response')
response=response_bundle.response
unfolding_result=unfold_bayes(RooUnfold,response_bundle,hStudiedMeasured,iterations=N_ITERATIONS,handle_fakes=True,name='hUnfoldedStudiedEtaCM')
unfold,hUnfoldedStudied,covariance=unfolding_result.algorithm,unfolding_result.histogram,unfolding_result.covariance
print(f'Controlled truth: {hControlledTruth.Integral():.6g}; measured: {hControlledMeasured.Integral():.6g}')
print(f'Studied truth: {hStudiedTruth.Integral():.6g}; measured: {hStudiedMeasured.Integral():.6g}')

## Flattened response and studied-sample closure


In [ ]:
canvas_response=draw_flattened_response(
    {'gen':('Controlled Gen',hControlledTruth),'reco':('Controlled Reco JER default',hControlledMeasured),'miss':('Controlled Miss',hControlledMiss),'fake':('Controlled Fake',hControlledFake)},
    hControlledResponse,annotations=(f'Controlled 60%, |#eta_{{CM}}| < {ETA_CUT:g}',),plot_miss_and_fakes=PLOT_MISS_AND_FAKES,
    output=OUTPUT_DIR/f'{OUTPUT_TAG}_flattened_response.pdf',save_png=SAVE_PNG,canvas_name='canvas_controlled_response')
canvas_closure,hStudiedMeasuredToTruth,hUnfoldedStudiedToTruth=draw_unfolding_closure(
    hStudiedTruth,hStudiedMeasured,hUnfoldedStudied,target_label='Studied Gen',target_role='gen',measured_label='Studied Reco JER default',
    x_title='global #eta_{CM} bin',ratio_range=(0.5,1.5),annotations=(f'Studied 40%, |#eta_{{CM}}| < {ETA_CUT:g}',),
    output=OUTPUT_DIR/f'{OUTPUT_TAG}_closure.pdf',save_png=SAVE_PNG,canvas_name='canvas_studied_closure',ratio_name_prefix='hStudiedClosure')
display(canvas_response);display(canvas_closure)

## Studied closure in every pTave interval


In [ ]:
(unfoldedStudiedByPt,studiedMeasuredToTruthByPt,unfoldedStudiedToTruthByPt,closureCanvasesByPt)=draw_unfolding_closure_by_pt(
    hUnfoldedStudied,studiedTruthByPt,studiedMeasuredByPt,layout,output_dir=OUTPUT_DIR,output_tag=OUTPUT_TAG,target_label='Studied Gen',
    target_role='gen',measured_label='Studied Reco JER default',eta_range=(-ETA_CUT-0.1,ETA_CUT+0.1),ratio_range=(0.5,1.5),
    annotation_prefix=('Studied 40%',),save_png=SAVE_PNG)
closureCanvasesByPt

## Save unfolding artifacts


In [ ]:
output_histograms=(hControlledTruth,hControlledMeasured,hStudiedTruth,hStudiedMeasured,hControlledMiss,hControlledFake,hControlledResponse,hMatchedControlledTruth,hMatchedControlledReco,hEffectiveControlledMiss,hEffectiveControlledFake,hBoundaryControlledMiss,hBoundaryControlledFake,hUnfoldedStudied,hStudiedMeasuredToTruth,hUnfoldedStudiedToTruth,*controlledTruthByPt,*controlledMeasuredByPt,*studiedTruthByPt,*studiedMeasuredByPt,*controlledMissByPt,*controlledFakeByPt,*unfoldedStudiedByPt,*studiedMeasuredToTruthByPt,*unfoldedStudiedToTruthByPt)
write_unfolding_output(OUTPUT_ROOT_FILE,histograms=output_histograms,covariance=covariance,covariance_name='hUnfoldedStudiedCovariance',response=response,response_name='controlledResponseEtaCM',metadata={'generator':GENERATOR,'direction':DIRECTION,'eta_cut':ETA_CUT,'pt_ave_bins':PT_AVE_INTERVALS,'ptave_bin_set':PTAVE_BIN_SET,'iterations':N_ITERATIONS,'response_scale':RESPONSE_SCALE,'split':'controlled60_studied40','handle_fakes':True})
print(f'Wrote {OUTPUT_ROOT_FILE}')